In [ ]:
import os

print(os.getcwd())

f:\Project\YTIntroModel\notebooks


In [ ]:
from pathlib import Path
import av
import numpy as np
import torch
import polars as pl

# Config
data_path = "../data/video_data.parquet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
df = pl.read_parquet(data_path)

df = df.filter(
    pl.col('media_type') == 'video',
    pl.col('channel_handle') != '@SecretBaseSBN',
    pl.col("view_count").is_not_null(),
    )

print(f"df shape: {df.shape}")

df shape: (2189, 24)


In [ ]:
# Get unique channel list
channels = df.select("channel_id").unique().to_series()

# List to collect per-channel normalized DataFrames
normalized_chunks = []

# Loop through channels and normalize
for ch in channels:
    
    channel_df = df.filter(pl.col("channel_id") == ch)
    # log scale view_count
    channel_df = channel_df.with_columns(
        pl.col("view_count").log().alias("log_view_count")
    )

    # set target to log view_count
    channel_df = channel_df.with_columns(
        pl.col("log_view_count").alias("target")
    )
    
    # Normalize view count by Z-score
    mean_vc = channel_df.select(pl.col("log_view_count").mean()).item()
    std_vc = channel_df.select(pl.col("log_view_count").std()).item()
    channel_df = channel_df.with_columns(
        ((pl.col("log_view_count") - mean_vc) / std_vc).alias("target")
    )

    # Add mean, std for debug
    channel_df = channel_df.with_columns(
        pl.lit(mean_vc).alias("mean_log_view_count"),
        pl.lit(std_vc).alias("std_log_view_count")
    )

    normalized_chunks.append(channel_df)

# Concatenate all normalized chunks
norm_df = pl.concat(normalized_chunks)
norm_df = norm_df.drop_nulls(subset=["target", "video_title"])

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:

import os

print(os.getcwd())

from pathlib import Path
import av
import numpy as np
import torch
import polars as pl

# Config
data_path = "../data/video_data.parquet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
df = pl.read_parquet(data_path)

df = df.filter(
    pl.col('media_type') == 'video',
    pl.col('channel_handle') != '@SecretBaseSBN',
    pl.col("view_count").is_not_null(),
    )

print(f"df shape: {df.shape}")

f:\Project\YTIntroModel\notebooks
df shape: (2189, 24)


In [ ]:
# Get unique channel list
channels = df.select("channel_id").unique().to_series()

# List to collect per-channel normalized DataFrames
normalized_chunks = []

# Loop through channels and normalize
for ch in channels:
    
    channel_df = df.filter(pl.col("channel_id") == ch)
    # log scale view_count
    channel_df = channel_df.with_columns(
        pl.col("view_count").log().alias("log_view_count")
    )

    # set target to log view_count
    channel_df = channel_df.with_columns(
        pl.col("log_view_count").alias("target")
    )
    
    # Normalize view count by Z-score
    mean_vc = channel_df.select(pl.col("log_view_count").mean()).item()
    std_vc = channel_df.select(pl.col("log_view_count").std()).item()
    channel_df = channel_df.with_columns(
        ((pl.col("log_view_count") - mean_vc) / std_vc).alias("target")
    )

    # Add mean, std for debug
    channel_df = channel_df.with_columns(
        pl.lit(mean_vc).alias("mean_log_view_count"),
        pl.lit(std_vc).alias("std_log_view_count")
    )

    normalized_chunks.append(channel_df)

# Concatenate all normalized chunks
norm_df = pl.concat(normalized_chunks)
norm_df = norm_df.drop_nulls(subset=["target", "video_title"])

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.BERT_TINY
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=25,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.53.0.dev0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

Freezing first 2 of 2 layers.
TitleRegressor with 128 hidden size and 4 layers.
Trainable: encoder.pooler.dense.weight - 16384 params
Trainable: encoder.pooler.dense.bias - 128 params
Trainable: regressor.0.weight - 16384 params
Trainable: regressor.0.bias - 128 params
Trainable: regressor.3.weight - 128 params
Trainable: regressor.3.bias - 1 params

Trainable parameters: 33,153 / 4,402,561


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        preds = model(input_ids, attention_mask)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 1 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                preds = model(input_ids, attention_mask)
                loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")

Training Epochs:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch 1/256 - Train Loss: 0.4782 | Val Loss: 0.3993 | Val MSE: 1.0071 | Val R2: -0.0122
Epoch 2/256 - Train Loss: 0.4522 | Val Loss: 0.4083 | Val MSE: 1.0459 | Val R2: -0.0512
Epoch 3/256 - Train Loss: 0.4562 | Val Loss: 0.4161 | Val MSE: 1.0684 | Val R2: -0.0738
Epoch 4/256 - Train Loss: 0.4473 | Val Loss: 0.4117 | Val MSE: 1.0513 | Val R2: -0.0566
Epoch 5/256 - Train Loss: 0.4408 | Val Loss: 0.4038 | Val MSE: 1.0227 | Val R2: -0.0279
Epoch 6/256 - Train Loss: 0.4352 | Val Loss: 0.3978 | Val MSE: 0.9988 | Val R2: -0.0039
Epoch 7/256 - Train Loss: 0.4226 | Val Loss: 0.3958 | Val MSE: 0.9863 | Val R2: 0.0087
Epoch 8/256 - Train Loss: 0.4241 | Val Loss: 0.3958 | Val MSE: 0.9814 | Val R2: 0.0136
Epoch 9/256 - Train Loss: 0.4167 | Val Loss: 0.3961 | Val MSE: 0.9793 | Val R2: 0.0158
Epoch 10/256 - Train Loss: 0.4204 | Val Loss: 0.3959 | Val MSE: 0.9779 | Val R2: 0.0172
Epoch 11/256 - Train Loss: 0.4120 | Val Loss: 0.3959 | Val MSE: 0.9780 | Val R2: 0.0171
Epoch 12/256 - Train Loss: 0.4144 |

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.T5_SMALL_YOUTUBE
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=25,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

tokenizer_config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "max_length": 300,
      "num_beams": 4,
      "prefix"

ValueError: Cannot locate transformer layers in the model

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)
        # print model architecture
        print(self.encoder)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.T5_SMALL_YOUTUBE
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=25,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 6,
  "num_heads": 8,
  "num_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "max_length": 300,
      "num_beams": 4,
      "prefix"

ValueError: Cannot locate transformer layers in the model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)
        # print model architecture
        print(self.encoder)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.DEBERTA_V3_XSMALL
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=25,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

ValueError: Converting from SentencePiece and Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast converters: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Tokenizer', 'Realm

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)
        # print model architecture
        print(self.encoder)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print(f"\nTrainable parameters: {trainable:,} / {total:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.YOUTUBE_XLM_ROBERTA_BASE
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=25,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Some weights of XLMRobertaModel were not initialized from the model checkpoint at AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "torch_dtype": "float32",
  "transformers_version": "4.53.0.dev0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        preds = model(input_ids, attention_mask)
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 1 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                preds = model(input_ids, attention_mask)
                loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")

Training Epochs:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch 1/256 - Train Loss: 0.4218 | Val Loss: 0.5771 | Val MSE: 1.5044 | Val R2: -0.5174
Epoch 2/256 - Train Loss: 0.5692 | Val Loss: 0.3878 | Val MSE: 0.9293 | Val R2: 0.0626
Epoch 3/256 - Train Loss: 0.4133 | Val Loss: 0.3897 | Val MSE: 0.9043 | Val R2: 0.0878
Epoch 4/256 - Train Loss: 0.4552 | Val Loss: 0.3999 | Val MSE: 0.9308 | Val R2: 0.0611
Epoch 5/256 - Train Loss: 0.4543 | Val Loss: 0.3876 | Val MSE: 0.9052 | Val R2: 0.0870
Epoch 6/256 - Train Loss: 0.4251 | Val Loss: 0.3833 | Val MSE: 0.9041 | Val R2: 0.0881
Epoch 7/256 - Train Loss: 0.4061 | Val Loss: 0.3867 | Val MSE: 0.9226 | Val R2: 0.0694
Epoch 8/256 - Train Loss: 0.4026 | Val Loss: 0.3918 | Val MSE: 0.9426 | Val R2: 0.0492
Epoch 9/256 - Train Loss: 0.4067 | Val Loss: 0.3948 | Val MSE: 0.9539 | Val R2: 0.0378
Epoch 10/256 - Train Loss: 0.4075 | Val Loss: 0.3950 | Val MSE: 0.9549 | Val R2: 0.0369
Epoch 11/256 - Train Loss: 0.4072 | Val Loss: 0.3927 | Val MSE: 0.9475 | Val R2: 0.0443
Epoch 12/256 - Train Loss: 0.4035 | Val 